In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# =========================================================================
# 1. ROBUST STATISTICAL TESTING FUNCTIONS (Self-Contained)
# =========================================================================

def local_chi_square_test(df, group_col, target_col, group_a_val, group_b_val):
    """ Runs a Chi-Squared test of independence for categorical targets """
    filtered_df = df[df[group_col].isin([group_a_val, group_b_val])].copy()
    
    if filtered_df[group_col].nunique() < 2:
        return {"test_type": "Chi-Squared", "p_value": None, "stat": None, "group_a_mean": None, "group_b_mean": None}
        
    contingency_table = pd.crosstab(filtered_df[group_col], filtered_df[target_col])
    
    # Handle empty contingency tables gracefully
    if contingency_table.size == 0 or (contingency_table.shape[0] < 2):
        return {"test_type": "Chi-Squared", "p_value": None, "stat": None, "group_a_mean": None, "group_b_mean": None}
        
    chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)
    rate_a = filtered_df[filtered_df[group_col] == group_a_val][target_col].mean()
    rate_b = filtered_df[filtered_df[group_col] == group_b_val][target_col].mean()
    
    return {"test_type": "Chi-Squared", "p_value": p_value, "stat": chi2, "group_a_mean": rate_a, "group_b_mean": rate_b}


def local_two_sample_t_test(df, group_col, target_col, group_a_val, group_b_val, conditional_col=None, conditional_val=None):
    """ Runs a Welch's Two-Sample independent t-test for numeric continuous targets """
    working_df = df.copy()
    
    if conditional_col is not None and conditional_val is not None:
        working_df = working_df[working_df[conditional_col] == conditional_val]
        
    group_a_data = working_df[working_df[group_col] == group_a_val][target_col].dropna()
    group_b_data = working_df[working_df[group_col] == group_b_val][target_col].dropna()
    
    if len(group_a_data) < 2 or len(group_b_data) < 2:
        return {"test_type": "Two-Sample T-Test", "p_value": None, "stat": None, "group_a_mean": None, "group_b_mean": None}
        
    t_stat, p_value = stats.ttest_ind(group_a_data, group_b_data, equal_var=False)
    
    return {"test_type": "Two-Sample T-Test", "p_value": p_value, "stat": t_stat, "group_a_mean": np.mean(group_a_data), "group_b_mean": np.mean(group_b_data)}

# =========================================================================
# 2. DATA TYPE SANITIZATION WORKFLOW
# =========================================================================

# Ensure targets are engineered correctly
df['Margin'] = df['TotalPremium'] - df['TotalClaims']
df['HasClaim'] = (df['TotalClaims'] > 0).astype(int)

# Force data types to align so filters find exact matches
df['Province'] = df['Province'].astype(str).str.strip()
df['Gender'] = df['Gender'].astype(str).str.strip()
df['ZipCode'] = df['ZipCode'].astype(str).str.strip()
df['Claimed'] = df['Claimed'].astype(int)

# Pull the exact string representation of your most frequent Zip Codes and Genders
top_zips = df['ZipCode'].value_counts().index.tolist()
zip_a, zip_b = top_zips[0], top_zips[1]

gender_labels = df['Gender'].unique().tolist()
gender_a, gender_b = gender_labels[0], gender_labels[1]

# =========================================================================
# 3. RUN ALL 4 HYPOTHESIS TESTS
# =========================================================================

results = {}

# H1: Risk Across Provinces (Claim Frequency)
results['Province_Risk'] = local_chi_square_test(
    df, group_col='Province', target_col='HasClaim', 
    group_a_val='Addis Ababa', group_b_val='Oromia'
)

# H2: Risk Between Zip Codes (Claim Frequency)
results['Zip_Risk'] = local_chi_square_test(
    df, group_col='ZipCode', target_col='HasClaim', 
    group_a_val=zip_a, group_b_val=zip_b
)

# H3: Margin Differences Between Zip Codes (Financial Margin)
results['Zip_Margin'] = local_two_sample_t_test(
    df, group_col='ZipCode', target_col='Margin', 
    group_a_val=zip_a, group_b_val=zip_b
)

# H4: Risk Differences Between Genders (Claim Severity)
results['Gender_Severity'] = local_two_sample_t_test(
    df, group_col='Gender', target_col='ClaimAmount', 
    group_a_val=gender_a, group_b_val=gender_b,
    conditional_col='Claimed', conditional_val=1
)

# =========================================================================
# 4. EXPLICIT PRINT SYSTEM FOR DELIVERABLES
# =========================================================================

print("\n" + "="*70)
print("             ACIS MARKETING ANALYTICS: HYPOTHESIS TESTING REPORT             ")
print("="*70)
print(f"DEBUG TARGETS INDENTIFIED:")
print(f" -> Provinces evaluated:  Group A = 'Addis Ababa' | Group B = 'Oromia'")
print(f" -> ZipCodes evaluated:   Group A = '{zip_a}' | Group B = '{zip_b}'")
print(f" -> Genders evaluated:    Group A = '{gender_a}' | Group B = '{gender_b}'")
print("-"*70)

for key, data in results.items():
    p = data.get('p_value')
    stat = data.get('stat')
    mean_a = data.get('group_a_mean')
    mean_b = data.get('group_b_mean')
    
    print(f"\n📊 TEST COMPONENT: {key.upper()}")
    print(f"  [Methodology]    {data.get('test_type')}")
    print(f"  [Test Statistic] {stat:.4f}" if stat is not None else "  [Test Statistic] None")
    print(f"  [Calculated p]   {p:.4e}" if p is not None else "  [Calculated p] None")
    print(f"  [Group A Baseline Average] {mean_a:.4f}" if mean_a is not None else "  [Group A Baseline Average] None")
    print(f"  [Group B Comparison Average] {mean_b:.4f}" if mean_b is not None else "  [Group B Comparison Average] None")
    
    if p is not None and not pd.isna(p):
        decision = "🔴 REJECT H0 (Statistically Significant Driver)" if p < 0.05 else "🟢 FAIL TO REJECT H0 (Identical Risk Profile)"
    else:
        decision = "⚠️ ABORTED: Check filter strings or segment sizes."
    print(f"  [MANAGEMENT DECISION] -> {decision}")

print("="*70 + "\n")

SyntaxError: unterminated string literal (detected at line 103) (2848174147.py, line 103)

In [2]:
# 1. Print column names to check casing
print("--- Actual Columns in Dataset ---")
print(df.columns.tolist())

# 2. Check if the newly engineered columns have data
print("\n--- Summary of Engineered Metrics ---")
if 'Margin' in df.columns and 'HasClaim' in df.columns:
    print(df[['Margin', 'HasClaim']].describe())
else:
    print("Columns 'Margin' and 'HasClaim' haven't been created yet!")

# 3. Check for Missing Values in your target columns
print("\n--- Missing Value Counts ---")
print(df[['TotalPremium', 'TotalClaims']].isnull().sum())

--- Actual Columns in Dataset ---
['CustomerID', 'Age', 'Gender', 'Province', 'VehicleType', 'AnnualIncome', 'RiskScore', 'AnnualPremium', 'Deductible', 'NCD', 'PastClaims', 'Claimed', 'ClaimAmount', 'TotalPremium', 'TotalClaims', 'CoverType', 'AutoMake', 'VehicleModel', 'CustomValueEstimate', 'ZipCode', 'TransactionDate', 'Margin', 'HasClaim']

--- Summary of Engineered Metrics ---
             Margin      HasClaim
count  10000.000000  10000.000000
mean    1173.939400      0.153500
std     3742.979999      0.360487
min   -44594.000000      0.000000
25%     1812.000000      0.000000
50%     2165.000000      0.000000
75%     2504.250000      0.000000
max     5079.000000      1.000000

--- Missing Value Counts ---
TotalPremium    0
TotalClaims     0
dtype: int64


In [4]:
print("Provinces found:", df['Province'].unique().tolist())
print("Genders found:", df['Gender'].unique().tolist())
print("Top 2 Zip Codes:\n", df['ZipCode'].value_counts().head(2))

Provinces found: ['Addis Ababa', 'Oromia', 'Somali', 'Tigray', 'Amhara']
Genders found: ['Male', 'Female']
Top 2 Zip Codes:
 ZipCode
10004    733
10002    732
Name: count, dtype: int64
